In [ ]:
import pandas as pd
import numpy as np
from datasets import load_dataset, get_dataset_config_names
import random

# =======================================================================
# 🏡 서울 아파트 전월세 거래 데이터 분석 실습
# 📋 데이터셋: kpubdata/seoul-apartment-rent
# ✨ 목표: 임장 전문가(Real Estate AI)가 되어, 아파트의 기본 정보(면적, 건축연도 등)만 가지고
#     '이 동네라면 이 정도 가격대일 거야!'라고 예측하는 간단한 데이터 탐색 실습을 해보는 것이 목표입니다!
#
# 이 데이터셋은 서울의 실제 아파트 월세/전세 거래 기록을 담고 있습니다.
# 우리는 이 데이터를 이용해서 아파트가 가진 시간적, 공간적 가치를 분석해 볼 거예요.
# =======================================================================

# --- 설정 상수 ---
DATASET_NAME = "kpubdata/seoul-apartment-rent"
SAMPLE_COUNT = 500 # 전체 데이터셋이 너무 크므로, 재미로 상위 500개 샘플만 분석할게요!

# -----------------------------------------------------------------------
# 1. 데이터셋 Config 확인 (필수 절차)
# -----------------------------------------------------------------------
print("✨ [튜터] AI 전문가가 되기 위한 첫걸음! 데이터셋 설정을 확인합니다.")

DATASET_ID = DATASET_NAME.split('/')[1] # kpubdata
try:
    configs = get_dataset_config_names(DATASET_ID)
    print(f"✅ 사용 가능한 Config 목록: {configs}")
    
    # 보통 기본 설정이 가장 안전합니다.
    selected_config = None 
except Exception as e:
    print("ℹ️ 해당 데이터셋은 별도의 Config가 없거나 기본 설정만 제공됩니다. (문제 없음!)")
    selected_config = None

# -----------------------------------------------------------------------
# 2. 데이터셋 로딩 (스트리밍 & 예외 처리)
# -----------------------------------------------------------------------
dataset = None
try:
    print("\n🚀 1단계: 초고속 스트리밍 모드로 데이터셋을 로드 시도합니다...")
    # ⭐️ 핵심: streaming=True를 사용하여 대용량 데이터셋의 메모리 부담을 줄이고,
    #         일단 메모리가 되는지 테스트해 봅니다.
    dataset = load_dataset(DATASET_NAME, split='train', streaming=True)
    print("✅ 스트리밍 로드 성공! 메모리 효율적으로 데이터를 읽어갈 준비가 되었습니다.")
except Exception as e:
    # 스트리밍 모드가 불안정하거나 환경 문제로 실패했을 경우를 대비합니다.
    print(f"⚠️ 스트리밍 로드 실패 ({e.__class__.__name__}). 일반 모드로 전환하여 진행합니다.")
    try:
        # 임시로 첫 100개의 샘플만 다운로드하여 진행 (메모리 절약)
        dataset = load_dataset(DATASET_NAME, split='train[:100]')
    except Exception as e_fallback:
        print(f"🚨 데이터 로딩에 실패했습니다. 환경을 확인해 주세요: {e_fallback}")
        exit()

# -----------------------------------------------------------------------
# 3. 샘플링 및 데이터 준비 (Iterator 활용)
# -----------------------------------------------------------------------
print("\n✨ [튜터] 다음은 데이터 전처리 시간! 상위 샘플만 골라 재미있게 분석해 봅시다.")

if hasattr(dataset, "take"):
    # .take()가 존재하면 스트리밍 데이터셋(IterableDataset)입니다.
    print(f"🔍 스트리밍 모드 감지! 상위 {SAMPLE_COUNT}개의 샘플을 불러옵니다.")
    sample_dataset_iterator = dataset.take(SAMPLE_COUNT)
else:
    # 일반 데이터셋 (Dataset)입니다.
    print(f"📚 일반 데이터셋 모드 감지! 상위 {SAMPLE_COUNT}개의 샘플을 불러옵니다.")
    # 만약 일반 데이터셋이고 size가 작으면 list()로 변환해도 무방합니다.
    sample_dataset_iterator = dataset.take(SAMPLE_COUNT)
    
# 샘플을 리스트로 한 번에 로드하여 분석하기 쉽게 만듭니다.
sample_data_list = []
for i, sample in enumerate(sample_dataset_iterator):
    sample_data_list.append(sample)

# 샘플 데이터를 Pandas DataFrame으로 변환합니다. (분석의 편리성을 높여줍니다!)
df = pd.DataFrame(sample_data_list)

print(f"✅ 데이터 준비 완료! 총 {len(df)}개의 샘플을 이용해 분석합니다.")

# -----------------------------------------------------------------------
# 4. 실습: 데이터의 특징 추출 (AI 예측 시뮬레이션)
# -----------------------------------------------------------------------
print("\n==============================================================")
print("🏠 🕵️‍♀️ 실습 시작: AI 임장 분석가 되기!")
print("==============================================================")

# 📌 4-1. 필수 피처 추출 및 전처리
# 분석에 사용할 핵심 숫자 데이터만 뽑아내고, 결측치나 이상한 값은 제거합니다.
core_features = ['exclusive_area', 'build_year', 'deal_year', 'district', 'monthly_rent', 'deposit']
df_clean = df[core_features].copy()

# '월세'나 '전세' 등의 범주형 데이터를 분석하기 좋게 그룹핑합니다.
# 월세와 전세의 특징을 종합적으로 볼 수 있도록 전월세의 중심 값만 사용합니다.
df_clean['is_jeonse'] = (df_clean['monthly_rent'] == 0).astype(int) 

# 💡 창의적인 가치 지표 생성: '건물 가치 점수'를 계산해봅시다!
# 건물 연식과 면적을 결합하여, 이 아파트가 시장에서 얼마나 가치 있게 평가될지 단순 계산합니다.
# (최신일수록, 클수록 가치 점수가 높다고 가정)
current_year = 2024 # 예측 기준 연도
df_clean['building_value_score'] = (
    df_clean['exclusive_area'] * (
        1 + (current_year - df_clean['build_year']) * 0.005
    )
)

print("✨ [튜터] '건물 가치 점수' (building_value_score) 생성 완료!")
print("   (면적 * (1 + (현재 연도 - 건축 연도) * 0.005)) -> 최신식, 대형 평수가 높은 점수를 받습니다.")


# 📌 4-2. 지역별 평균 분석 (가장 쉬우면서도 효과적인 AI 탐색!)
print("\n🏆 2단계: 지역별(District) '평균 가치' 비교 분석!")
print("   특정 동네가 다른 동네보다 월세나 전세 가격이 평균적으로 높은지 확인해봅시다.")

# 지리적 특성이 가격에 미치는 영향을 분석하기 위해 평균을 구합니다.
# 여기서 'Mean'을 구하는 행위 자체가 AI가 데이터를 요약하고 특징을 파악하는 과정을 흉내냅니다.
avg_analysis = df_clean.groupby('district')[['monthly_rent', 'deposit', 'building_value_score']].mean().sort_values(by='monthly_rent', ascending=False)

print("\n--- 📊 지역별 평균 임대료 분석 (가장 비싼 동네 Top 5) ---")
print(avg_analysis.head())

# 📌 4-3. '규모 대비 효율' 확인 (가격 적정성 탐색)
print("\n--- 📐 3단계: 아파트 규모 대비 가격 효율성 탐색 ---")
# 임대료(월세+전세)를 면적(exclusive_area)으로 나눈 '㎡당 평균 비용'을 계산합니다.
# 이 값이 낮을수록, 같은 면적 대비 상대적으로 '가성비'가 좋다고 판단할 수 있습니다!
df_clean['cost_per_sqm'] = (df_clean['monthly_rent'] + df_clean['deposit']) / df_clean['exclusive_area']

# 가장 비용 효율이 좋아 보이는 아파트를 찾아봅시다.
best_value_apartments = df_clean.sort_values(by='cost_per_sqm').head(5)

print("\n💖 가장 가성비가 좋아 보이는 아파트 TOP 5 (낮은 '㎡당 평균 비용' 순)")
# 소수점 몇 자리만 보여서 보기 좋게 만듭니다.
print(best_value_apartments[['exclusive_area', 'monthly_rent', 'deposit', 'cost_per_sqm']].round(0).to_markdown())


# -----------------------------------------------------------------------
# 5. 튜터의 코멘트: 이 코드가 무엇을 했나요?
# -----------------------------------------------------------------------
print("\n==============================================================")
print("✨ [튜터 마무리] 오늘 무엇을 배웠을까요?")
print("==============================================================")
print("1. 💾 **데이터 로딩:** 대용량 데이터를 처리하기 위해 `streaming=True` 패턴을 배웠습니다.")
print("2. 🔢 **Feature Engineering:** 단순히 주어진 숫자 외에, `건물 가치 점수`나 `㎡당 비용`처럼")
print("   우리가 직접 가설을 세워 새로운 지표를 만드는 법(이것이 AI의 핵심 사고방식!)을 경험했어요.")
print("3. 🧐 **탐색적 분석:** `groupby()` 함수를 사용하여 '지역'이라는 기준으로 묶어")
print("   지역 간의 평균 차이(예: 강남 vs. 구도심)가 가격에 얼마나 영향을 주는지 분석할 수 있습니다.")
print("\n🎉 축하합니다! 이제 당신은 데이터의 숨겨진 가치(잠재력)를 찾아내는 기초 분석가입니다!")